# 04 — Walk-forward GBM (Case B)

Expanding-window global `HistGradientBoostingRegressor`:

- train on `date < t`, score `date == t` for each Tuesday `t ≥ 2025-01-07`
- features via `LaneWeekFeatureBuilder` (lagged RPM + `availability_lag_1`; no same-week availability / rate bands)
- baseline = `rpm_lag_1`

Prefer the reproducible script for full runs:

```bash
python scripts/train_walkforward_gbm.py
```

This notebook loads saved metrics (or re-runs) and plots weekly MAE.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from freight_rates.ingestion import load_raw_snapshot
from freight_rates.preprocessing import build_lane_week_panel
from freight_rates.splits import filter_model_window
from freight_rates.walkforward import run_walkforward_gbm, save_walkforward_outputs

ROOT = Path("..").resolve()
OUT = ROOT / "models" / "walkforward_gbm"
PRED_PATH = OUT / "walkforward_gbm_predictions.parquet"
WEEK_PATH = OUT / "walkforward_gbm_metrics_by_week.csv"
HIST_PATH = OUT / "walkforward_gbm_metrics_by_history.csv"
OVERALL_PATH = OUT / "walkforward_gbm_metrics_overall.csv"

In [ ]:
if PRED_PATH.exists() and WEEK_PATH.exists():
    preds = pd.read_parquet(PRED_PATH)
    by_week = pd.read_csv(WEEK_PATH, parse_dates=["date"])
    by_history = pd.read_csv(HIST_PATH)
    overall = pd.read_csv(OVERALL_PATH)
    print("Loaded saved walk-forward outputs from", OUT)
else:
    print("No saved outputs — running walk-forward (may take a few minutes)...")
    panel = filter_model_window(build_lane_week_panel(load_raw_snapshot(raw_dir=ROOT / "data" / "raw")))
    result = run_walkforward_gbm(panel, verbose=True)
    save_walkforward_outputs(result, OUT)
    preds, by_week, by_history, overall = (
        result.predictions,
        result.by_week,
        result.by_history,
        result.overall,
    )

display(overall)
display(by_history)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(by_week["date"], by_week["mae"], label="GBM MAE")
if "mae_baseline" in by_week.columns:
    ax.plot(by_week["date"], by_week["mae_baseline"], label="lag-1 baseline MAE", alpha=0.8)
ax.set_title("Weekly MAE — expanding-window walk-forward")
ax.set_ylabel("MAE ($/mile)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"scored rows: {len(preds):,}")